In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings
from matplotlib.collections import PathCollection
from matplotlib.legend_handler import HandlerPathCollection, HandlerLine2D







def update(handle, orig):
    handle.update_from(orig)
    handle.set_alpha(1)
    handle.set_markersize(3)


In [ ]:
def B_from_timing(P: float, Pdot: float) -> float:
    """
    B field estimated from timing properties. 
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): value of the dipolar component of the magnetic field at the
        magnetic pole for a simulated neutron star, measured in [G].
    """

    # Period derivative.
    B = np.sqrt(P * Pdot / (4 * np.pi**2 * beta_1))

    return B

def Edot_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar rotational power loss from timing properties. 
    
    Args:
        P (float): spin period of a simulated pulsar, measured in [s].
        Pdot (float): period derivative of a simulated pulsar in [s/s].   

    Returns:
        (float): characteristic age in [yr].     
    """

    # Period derivative.
    Erot_dot = (2.*np.pi)**2 * NS_inertia * Pdot / P**3

    return Erot_dot




# Characteristic neutron star radius in [cm].
NS_radius = cfg["NS_radius"] #7.e8
# Characteristic neutron star mass in solar masses.
NS_mass = cfg["NS_mass"]
# Dimensionless coefficients k_0, k_1, k_2 for a force-free magnetosphere
# taken from Spitkovsky (2006) and Philippov et al. (2014).
# For comparison, in vacuum k_0 = 0 and k_1 = k_2 = 2/3.
k_coefficients = [1.0, 1.0, 1.0]
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * NS_mass * NS_radius ** 2
# Auxiliary quantity beta as defined in eq. (72) of Pons & Vigano (2019).
beta = 1./4. * NS_radius ** 6 / (NS_inertia * const.C ** 3)
#beta = np.pi ** 2 * NS_radius ** 6 / (NS_inertia * const.c ** 3)
print(beta)
# Assume an inclination angle in [rad].
chi = 0.
beta_1 = beta * (k_coefficients[0] + k_coefficients[1] * np.sin(chi) ** 2)


In [ ]:
#GLEAM:
P_gleam = 1091 #s
Pdot_gleam = 1.5e-9 #s
Edot_gleam  = 6.17e28 #erg/s
Lum_gleam = 4e31 #erg/s
#GPM:
P_gpm = 1318 #s
Pdot_gpm = 3.6e-13 #s
Edot_gpm  = 8.4e24 #erg/s
Lum_gpm =1e28 #erg/s

In [ ]:
Edot_from_timing(P_gleam,Pdot_gleam)

In [ ]:
Edot_from_timing(P_gpm,Pdot_gpm)

## Deathlines assuming the beta = 10 for the fourth

In [ ]:
def first_chen(P,R,I):
    C = 2.34e106
    C_B = 2.02e15
    Pdot = np.sqrt((P**(5.5))*C*(R**(2.5))*(C_B**(-4))*(I**(-2)))
    return Pdot

def fourth_chen(P,M):
    C = 7.124e90
    C_B = 2.02e15
    C_I = (2/5)**(-2)
    Pdot = np.sqrt((P**(4))*C*(C_B**(-4))*(M**(-2))*C_I)
    return Pdot

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "/home/celsa/Documents/Jupyter-notebooks/data/atnf_full_nobinary_25-04-2023.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
# Select only stars with measured P, Pdot, DM and radio flux.
# We also select only those that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["P0"]["[s]"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["P1"]["[s/s]"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["DM"]["[cm^-3pc]"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["DIST"]["[kpc]"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["S1400"]["[mJy]"].isin(["NAN"])]
df_atnf = df_atnf[~df_atnf["ASSOC"]["Unnamed: 24_level_1"].str.match("|".join(discard))]

# select only isolated non recycled neutron stars i.e. with Pdot > 1e-19
df_atnf = df_atnf[df_atnf["P1"]["[s/s]"].to_numpy().astype(np.float64) > 1.0e-19]
P_atnf = df_atnf["P0"]["[s]"].to_numpy().astype(np.float64)
P_dot_atnf = df_atnf["P1"]["[s/s]"].to_numpy().astype(np.float64)


In [ ]:
data_i_1 = pd.read_pickle(
    "../toremove/nanda_experiment/NS1_Bdecay/initial_population.pkl.gz",
    compression="gzip",
)

data_f_1 = pd.read_pickle(
    "../toremove/nanda_experiment/NS1_Bdecay/final_population.pkl.gz",
    compression="gzip",
)


In [ ]:
data_i_3 = pd.read_pickle(
    "../toremove/nanda_experiment/NS2_Bconst/initial_population.pkl.gz",
    compression="gzip",
)

data_f_3 = pd.read_pickle(
    "../toremove/nanda_experiment/NS2_Bconst/final_population.pkl.gz",
    compression="gzip",
)


In [ ]:
P_i_1 = data_i_1["P"]["[s]"].to_numpy()
P_dot_i_1 = data_i_1["P_dot"]["[s s^-1]"].to_numpy()
B_i_1 = data_i_1["B"]["[G]"].to_numpy()


P_f_1 = data_f_1["P"]["[s]"].to_numpy()
P_dot_f_1 = data_f_1["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol_1 = data_f_1["L_radio_bol"]["[erg s^-1]"].to_numpy()
B_f_1 = data_f_1["B"]["[G]"].to_numpy()

m = cfg["NS_mass"]
R = cfg["NS_radius"]
I = (2/5)*m*R**2
Erot_f_1 = ((2*np.pi)**2)*I*(P_dot_f_1/P_f_1**3)

intercept_radio_1 = data_f_1[('intercepted_radio',' ')] == 1
data_f_1[intercept_radio_1].head()

In [ ]:
P_atnf

In [ ]:
P_i_3 = data_i_3["P"]["[s]"].to_numpy()
P_dot_i_3 = data_i_3["P_dot"]["[s s^-1]"].to_numpy()
B_i_3 = data_i_3["B"]["[G]"].to_numpy()


P_f_3 = data_f_3["P"]["[s]"].to_numpy()
P_dot_f_3 = data_f_3["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol_3 = data_f_3["L_radio_bol"]["[erg s^-1]"].to_numpy()
B_f_3 = data_f_3["B"]["[G]"].to_numpy()

m = cfg["NS_mass"]
R = cfg["NS_radius"]
I = (2/5)*m*R**2

Erot_f_3 = ((2*np.pi)**2)*I*(P_dot_f_3/P_f_3**3)


Erot_atnf = ((2*np.pi)**2)*I*(P_dot_atnf/P_atnf**3)


intercept_radio_3 = data_f_3[('intercepted_radio',' ')] == 1

In [ ]:
from matplotlib.ticker import ScalarFormatter

P_bins = np.logspace(-2, 6, 31)

figsize=(12, 10)
# Create the figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})



# -----------------------Plot the distribution of P_i_1 at the top-----------------

ax1.hist(P_i_1, bins=P_bins,
    histtype="step",
    edgecolor="lightgrey",
    lw=3,)

ax1.hist(P_f_1[intercept_radio_1],
     bins=P_bins,
     histtype="step",
     edgecolor="black",
         lw=3,
        )

ax1.hist(P_f_1,
     bins=P_bins,
     histtype="step",
     edgecolor="dimgrey",
         lw=3,
         ls= '--'
        )

ax1.hist(P_atnf,
     bins=P_bins,
     histtype="step",
     edgecolor="gold",
         lw=3,
         ls= '--'
        )
ax1.text(9e3,1e5,'NS1_Bdecay',fontsize = 35)

ax1.set_yscale('log')
ax1.set_yticks([1e2, 1e4, 1e6])
ax1.set_ylabel("# NSs")
ax1.tick_params(axis="x", labelbottom=False)
#ax1.legend(markerscale=0,loc='upper right')
plt.xscale('log') 
plt.yscale('log') 

#------------------------Plot the B and Edot lines--------------------------------
P_min = 1e-3
P_max = 1e7
Pdot_min = 1e-28
Pdot_max = 1e-2

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')
B_timing = B_from_timing(P_grid, Pdot_grid)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([6.,9.,12.,15.,18]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,
    #interpolation='none'
)

fmt = {}
strs = ['$10^{6}$ G', '$10^{9}$ G', '$10^{12}$ G', '$10^{15}$ G','$10^{17}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"

manual_locations = [(8e-3, 1e-26), (8e-3, 1e-18), (5e-3, 1e-12), (4e-3, 1e-6), (1e0, 1e-4)]

ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)


#-------------------------------------Edot constant lines-----------------------------------------
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

contour_Edot = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(Edot_timing),
    levels = np.array([12.,20.,29.,38.,47.]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,

    #interpolation='none'
)

fmt = {}
strs = ['$10^{12}$ erg s$^{-1}$', '$10^{20}$ erg s$^{-1}$', '$10^{29}$ erg s$^{-1}$', '$10^{38}$ erg s$^{-1}$', '$10^{47}$ erg s$^{-1}$']

for l,s in zip( contour_Edot.levels, strs ):
    fmt[l] = rf"{s}"
    
manual_locations = [(1e6, 1e-20), (1e6, 1e-12), (1e4, 1e-6), (1e1, 1e-5), (9e-2, 1e-6)]

ax2.clabel(
    contour_Edot, 
    contour_Edot.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20,
    colors='black'
)



#----------------------------Plot of the points in the 2D-------------------------------------


ax2.plot(
    P_i_1,
    P_dot_i_1,
    linestyle="None",
    marker="o",
    color="lightgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Initial population'
)

ax2.plot(
    P_f_1,
    P_dot_f_1,
    linestyle="None",
    marker="o",
    color="dimgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population'
)



ax2.plot(
    P_f_1[intercept_radio_1],
    P_dot_f_1[intercept_radio_1],
    linestyle="None",
    marker="o",
    color="black",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los'
)
ax2.plot(
    P_atnf,
    P_dot_atnf,
    linestyle="None",
    marker='o', 
    markersize=1.5, 
    markerfacecolor='gold',
    markeredgecolor='gold',
    alpha = 0.3,
    rasterized=True,
    label = 'Observed radio pulsars'
)    
ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$ [s s$^{-1}$]")
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(P_min,P_max)
ax2.set_ylim(Pdot_min,Pdot_max)
plt.legend(frameon=False, loc='lower right', fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

# Remove the space between the subplots
plt.subplots_adjust(hspace=0)
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_top_right.png',format= 'png')

# Show the plot
plt.show()


In [ ]:
from matplotlib.ticker import ScalarFormatter

P_bins = np.logspace(-2, 7, 31)
P_deadlines = np.logspace(-3, 7, 31)
NS_inertia = 2.0 / 5.0 * NS_mass * NS_radius ** 2
Pdot_deadlines_I = first_chen(P_deadlines,NS_radius,NS_inertia)
Pdot_deadlines_IV = fourth_chen(P_deadlines,NS_mass)

# ---------------------- Deadlines ------------


figsize=(12, 10)
# Create the figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})



# -----------------------Plot the distribution of P_i_1 at the top-----------------

ax1.hist(P_i_1, bins=P_bins,
    histtype="step",
    edgecolor="lightgrey",
    lw=3,)

ax1.hist(P_f_1[intercept_radio_1],
     bins=P_bins,
     histtype="step",
     edgecolor="black",
         lw=3,
        )

ax1.hist(P_f_1,
     bins=P_bins,
     histtype="step",
     edgecolor="dimgrey",
         lw=3,
         ls= '--'
        )

ax1.hist(P_atnf,
     bins=P_bins,
     histtype="step",
     edgecolor="gold",
         lw=3,
         ls= '--'
        )
ax1.text(9e3,1e5,'NS1_Bdecay',fontsize = 35)

ax1.set_yscale('log')
ax1.set_yticks([1e2, 1e4, 1e6])
ax1.set_ylabel("# NSs")
ax1.tick_params(axis="x", labelbottom=False)
#ax1.legend(markerscale=0,loc='upper right')
plt.xscale('log') 
plt.yscale('log') 

#------------------------Plot the B and Edot lines--------------------------------
P_min = 1e-3
P_max = 1e7
Pdot_min = 1e-28
Pdot_max = 1e-2

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')
B_timing = B_from_timing(P_grid, Pdot_grid)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([6.,9.,12.,15.,18]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,
    #interpolation='none'
)

fmt = {}
strs = ['$10^{6}$ G', '$10^{9}$ G', '$10^{12}$ G', '$10^{15}$ G','$10^{17}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"

manual_locations = [(8e-3, 1e-26), (8e-3, 1e-18), (5e-3, 1e-12), (4e-3, 1e-6), (1e0, 1e-4)]

ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)




#----------------------------Plot of the points in the 2D-------------------------------------

ax2.plot(
    P_i_1,
    P_dot_i_1,
    linestyle="None",
    marker="o",
    color="lightgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Initial population'
)

ax2.plot(
    P_f_1,
    P_dot_f_1,
    linestyle="None",
    marker="o",
    color="dimgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population'
)



ax2.plot(
    P_f_1[intercept_radio_1],
    P_dot_f_1[intercept_radio_1],
    linestyle="None",
    marker="o",
    color="black",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los'
)
ax2.plot(
    P_atnf,
    P_dot_atnf,
    linestyle="None",
    marker='o', 
    markersize=1.5, 
    markerfacecolor='gold',
    markeredgecolor='gold',
    alpha = 0.3,
    rasterized=True,
    label = 'Observed radio pulsars'
) 
#----------------------------Plot the deadlines-------------------------------------

ax2.plot(
    P_deadlines,
    Pdot_deadlines_I,
    rasterized=True,
    c = 'black',
    linewidth = 2,
    alpha = 0.3,
    linestyle = '--',
)
ax2.plot(
    P_deadlines,
    Pdot_deadlines_IV,
    rasterized=True,
     c = 'black',
    linewidth = 2,
    alpha = 0.3,
)

#--------------------GLEAM-X and GPM--------------
ax2.plot(
    P_gleam,
    Pdot_gleam,
    linestyle="None",
    marker="^",
    color="black",
    markersize=10,
    alpha=1,
    rasterized=True,
)
ax2.annotate(
    "", 
    xy=(P_gleam, Pdot_gleam/100), 
    xytext=(P_gleam, Pdot_gleam), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax2.text(P_gleam*1.5, Pdot_gleam, "GLEAM-X J1627", color='black', fontsize=20)

ax2.plot(
    P_gpm,
    Pdot_gpm,
    linestyle="None",
    marker="*",
    color="black",
    markersize=15,
    alpha=1,
    rasterized=True,
)
ax2.annotate(
    "", 
    xy=(P_gpm, Pdot_gpm/100), 
    xytext=(P_gpm, Pdot_gpm), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax2.text(P_gpm*1.5, Pdot_gpm, " GPM J1839–10", color="black", fontsize=20)

ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$ [s s$^{-1}$]")
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(P_min,P_max)
ax2.set_ylim(Pdot_min,Pdot_max)
plt.legend(frameon=False, loc='lower right', fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

# Remove the space between the subplots
plt.subplots_adjust(hspace=0)
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_top_right.png',format= 'png')

# Show the plot
plt.show()


In [ ]:
from matplotlib.ticker import ScalarFormatter

P_bins = np.logspace(-2, 7, 31)
P_deadlines = np.logspace(-3, 7, 31)
NS_inertia = 2.0 / 5.0 * NS_mass * NS_radius ** 2
Pdot_deadlines_I = first_chen(P_deadlines,NS_radius,NS_inertia)
Pdot_deadlines_IV = fourth_chen(P_deadlines,NS_mass)

P_bins = np.logspace(-2, 6, 31)

figsize=(12, 10)
# Create the figure and axes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [1, 3], "hspace": 0})



# -----------------------Plot the distribution of P_i_1 at the top-----------------

ax1.hist(P_i_3, bins=P_bins,
    histtype="step",
    edgecolor="grey",
    lw=3,)

ax1.hist(P_f_3[intercept_radio_3],
     bins=P_bins,
     histtype="step",
     edgecolor="seagreen",
         lw=3,
        )

ax1.hist(P_f_3,
     bins=P_bins,
     histtype="step",
     edgecolor="lightgreen",
         lw=3,
         ls= '--'
        )
ax1.hist(P_atnf,
     bins=P_bins,
     histtype="step",
     edgecolor="gold",
         lw=3,
         ls= '--'
        )
ax1.text(9e3,1e5,'NS2_Bconst',fontsize = 35)

ax1.set_yscale('log')
ax1.set_yticks([1e2, 1e4, 1e6])
ax1.set_ylabel("# NSs")
ax1.tick_params(axis="x", labelbottom=False)
#ax1.legend(markerscale=0,loc='upper right')
plt.xscale('log') 
plt.yscale('log') 

#------------------------Plot the B and Edot lines--------------------------------
P_min = 1e-3
P_max = 1e7
Pdot_min = 1e-28
Pdot_max = 1e-2

log_P_edges = np.linspace(np.log10(P_min), np.log10(P_max), 71)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(np.log10(Pdot_min), np.log10(Pdot_max), 71)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing='ij')
B_timing = B_from_timing(P_grid, Pdot_grid)

contour_B = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(B_timing), 
    levels = np.array([6.,9.,12.,15.,18]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,
    #interpolation='none'
)

fmt = {}
strs = ['$10^{6}$ G', '$10^{9}$ G', '$10^{12}$ G', '$10^{15}$ G','$10^{17}$ G']
for l,s in zip( contour_B.levels, strs ):
    fmt[l] = rf"{s}"

manual_locations = [(8e-3, 1e-26), (8e-3, 1e-18), (5e-3, 1e-12), (4e-3, 1e-6), (1e0, 1e-4)]

ax2.clabel(
    contour_B, 
    contour_B.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20, 
    colors='black'
)

'''
#-------------------------------------Edot constant lines-----------------------------------------
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

contour_Edot = ax2.contour(
    P_grid, 
    Pdot_grid,
    np.log10(Edot_timing),
    levels = np.array([12.,20.,29.,38.,47.]), 
    colors='black',
    linestyles='dashed',
    alpha = 0.7,

    #interpolation='none'
)

fmt = {}
strs = ['$10^{12}$ erg s$^{-1}$', '$10^{20}$ erg s$^{-1}$', '$10^{29}$ erg s$^{-1}$', '$10^{38}$ erg s$^{-1}$', '$10^{47}$ erg s$^{-1}$']

for l,s in zip( contour_Edot.levels, strs ):
    fmt[l] = rf"{s}"
    
manual_locations = [(1e6, 1e-20), (1e6, 1e-12), (1e4, 1e-6), (1e1, 1e-5), (9e-2, 1e-6)]

ax2.clabel(
    contour_Edot, 
    contour_Edot.levels, 
    inline=True, 
    manual=manual_locations, 
    fmt=fmt, 
    fontsize=20,
    colors='black'
)
'''



#----------------------------Plot of the points in the 2D-------------------------------------

    
ax2.plot(
    P_i_3,
    P_dot_i_3,
    linestyle="None",
    marker="o",
    color="lightgrey",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Initial population'
)

ax2.plot(
    P_f_3,
    P_dot_f_3,
    linestyle="None",
    marker="o",
    color="lightgreen",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population'
)



ax2.plot(
    P_f_3[intercept_radio_3],
    P_dot_f_3[intercept_radio_3],
    linestyle="None",
    marker="o",
    color="seagreen",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los'
)
ax2.plot(
    P_atnf,
    P_dot_atnf,
    linestyle="None",
    marker='o', 
    markersize=1.5, 
    markerfacecolor='gold',
    markeredgecolor='gold',
    alpha = 0.3,
    rasterized=True,
    label = 'Observed radio pulsars'
)  

#----------------------------Plot the deadlines-------------------------------------

ax2.plot(
    P_deadlines,
    Pdot_deadlines_I,
    rasterized=True,
    c = 'black',
    linewidth = 2,
    alpha = 0.3,
    linestyle = '--',
)
ax2.plot(
    P_deadlines,
    Pdot_deadlines_IV,
    rasterized=True,
     c = 'black',
    linewidth = 2,
    alpha = 0.3,
)

#--------------------GLEAM-X and GPM--------------
ax2.plot(
    P_gleam,
    Pdot_gleam,
    linestyle="None",
    marker="^",
    color="black",
    markersize=10,
    alpha=1,
    rasterized=True,
)
ax2.annotate(
    "", 
    xy=(P_gleam, Pdot_gleam/100), 
    xytext=(P_gleam, Pdot_gleam), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax2.text(P_gleam*1.5, Pdot_gleam, "GLEAM-X J1627", color='black', fontsize=20)

ax2.plot(
    P_gpm,
    Pdot_gpm,
    linestyle="None",
    marker="*",
    color="black",
    markersize=15,
    alpha=1,
    rasterized=True,
)
ax2.annotate(
    "", 
    xy=(P_gpm, Pdot_gpm/100), 
    xytext=(P_gpm, Pdot_gpm), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax2.text(P_gpm*1.5, 1e-12, " GPM J1839–10", color="black", fontsize=20)

ax2.set_xlabel(r"$P$ [s]")
ax2.set_ylabel(r"$\dot{P}$ [s s$^{-1}$]")
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(P_min,P_max)
ax2.set_ylim(Pdot_min,Pdot_max)
plt.legend(frameon=False, loc='lower right', fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})

# Remove the space between the subplots
plt.subplots_adjust(hspace=0)
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_top_right.png',format= 'png')

# Show the plot
plt.show()


In [ ]:
P_bins = np.logspace(-2, 6, 31)

#P_bins_shifted = np.roll(P_bins, 1)
#size_bin = P_bins - P_bins_shifted

fig, ax = plt.subplots(figsize=(12,10))
#hist_P_i, bin_edges_P_i = np.histogram(P_i, bins = P_bins)


ax.hist(
    P_f_1,
    bins=P_bins,
    histtype="step",
    edgecolor="dimgrey",
    cumulative = True,
    lw=3,
    ls="--",
    label="Final population NS1_Bdecay",
)
ax.hist(
    P_f_3,
    bins=P_bins,
    histtype="step",
    edgecolor="lightgreen",
    cumulative = True,
    lw=3,
    ls="--",
    label="Final population NS2_Bconst",
)
ax.hist(
    P_f_1[intercept_radio_1],
    bins=P_bins,
    histtype="step",
    edgecolor="black",
    cumulative = True,
    lw=3,
    ls="-",
    label="Intercept our los NS1_Bdecay",
)

ax.hist(
    P_f_3[intercept_radio_3],
    bins=P_bins,
    histtype="step",
    edgecolor="seagreen",
    cumulative = True,
    lw=3,
    ls="-",
    label="Intercept our los NS2_Bconst",
)






plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Cumulative # NSs")
#plt.xlim(1e0,1e9)
plt.xlim(1e-3, 1e6)
plt.xscale('log')
plt.yscale('log')
plt.legend(frameon=False, loc=4, fontsize=25)
plt.grid()
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_bottom_left.png',format= 'png')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))



ax.plot(
    P_f_1,
    L_radio_bol_1,
    linestyle="None",
    marker="o",
    color="dimgrey",
    markersize=3,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS1_Bdecay'
)

ax.plot(
    P_f_3,
    Erot_f_3,
    linestyle="None",
    marker="o",
    color="lightgreen",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS2_Bconst'
)

ax.plot(
    P_f_1[intercept_radio_1],
    L_radio_bol_1[intercept_radio_1],
    linestyle="None",
    marker="o",
    color="black",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS1_Bdecay'
)





ax.plot(
    P_f_3[intercept_radio_3],
    Erot_f_3[intercept_radio_3],
    linestyle="None",
    marker="o",
    color="seagreen",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS2_Bconst'
)


#plt.axvline(10**(-0.7))
ax.set_ylabel(r"$L_{\rm radio} $")
ax.set_xlabel(r"$P$ [s]")
plt.xscale('log') 
plt.yscale('log') 
plt.xlim(1e-3,1e6)
#plt.ylim(1e-30,1e-12)
#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5)
plt.legend(frameon=False, loc=0, fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})
plt.grid()

#plt.savefig()
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_bottom_right.png',format= 'png')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))



ax.plot(
    P_f_1,
    Erot_f_1,
    linestyle="None",
    marker="o",
    color="dimgrey",
    markersize=3,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS1_Bdecay'
)

ax.plot(
    P_f_3,
    Erot_f_3,
    linestyle="None",
    marker="o",
    color="lightgreen",
    markersize=2,
    alpha=0.5,
    rasterized=True,
    label = 'Final population NS2_Bconst'
)







ax.plot(
    P_f_3[intercept_radio_3],
    Erot_f_3[intercept_radio_3],
    linestyle="None",
    marker="o",
    color="seagreen",
    markersize=0.5,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS2_Bconst'
)
ax.plot(
    P_f_1[intercept_radio_1],
    Erot_f_1[intercept_radio_1],
    linestyle="None",
    marker="o",
    color="black",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label = 'Intercept our los NS1_Bdecay'
)
ax.plot(
    P_atnf,
    Erot_atnf,
    linestyle="None",
    marker='o', 
    markersize=1.5, 
    markerfacecolor='gold',
    markeredgecolor='gold',
    alpha = 0.3,
    rasterized=True,
    label = 'Observed radio pulsars'
)

#----------------------GPM and GLEAM -------------------------------
ax.plot(
    P_gleam,
    Edot_gleam,
    linestyle="None",
    marker="^",
    color="black",
    markersize=10,
    alpha=1,
    rasterized=True,
)
ax.plot(
    P_gleam,
    Lum_gleam,
    linestyle="None",
    marker="^",
    color="black",
    markersize=10,
    alpha=1,
    rasterized=True,
)
ax.annotate(
    "", 
    xy=(P_gleam, Edot_gleam/100), 
    xytext=(P_gleam, Edot_gleam), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax.text(P_gleam*1.4, 1e29, "GLEAM-X J1627", color='black', fontsize=20)

ax.plot(
    P_gpm,
    Edot_gpm,
    linestyle="None",
    marker="*",
    color="black",
    markersize=15,
    alpha=1,
    rasterized=True,
)
ax.plot(
    P_gpm,
    Lum_gpm,
    linestyle="None",
    marker="*",
    color="black",
    markersize=15,
    alpha=1,
    rasterized=True,
)
ax.annotate(
    "", 
    xy=(P_gpm, Edot_gpm/100), 
    xytext=(P_gpm, Edot_gpm), 
    arrowprops=dict(arrowstyle="->", lw=2, color='black', zorder=-1), 
    alpha=1
) 
ax.text(P_gpm*1.5, Edot_gpm, " GPM J1839–10", color='black', fontsize=20)

#plt.axvline(10**(-0.7))
ax.set_ylabel(r"$\dot{E} \, [\rm erg \, s^{-1}]$")
ax.set_xlabel(r"$P$ [s]")
plt.xscale('log') 
plt.yscale('log') 
plt.xlim(1e-3,1e7)
plt.ylim(1e5,1e40)
#plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20,markerscale=5
plt.legend(frameon=False,loc = 'lower left',  fontsize=25,markerscale=5, handler_map={PathCollection : HandlerPathCollection(update_func= update),
                        plt.Line2D : HandlerLine2D(update_func = update)})
plt.grid()

#plt.savefig()
#plt.savefig('/home/celsa/Documents/paper_nandaetal_2023/figure1_top_right.png',format= 'png')

plt.show()